# BSM implied volatility by bisection

시장가격 $V_{market}$에 대해 다음 scalar 방정식의 해를 찾는다.

$$
f(\sigma) = V_{BSM}(\sigma) - V_{market} = 0
$$

유럽형 옵션의 BSM 가격은 변동성에 대해 단조 증가하므로, 해가 설정한 bracket 안에 있으면 이분탐색으로 bracket을 반복해서 절반으로 줄일 수 있다. 구현은 기존 `bsm_price`를 재사용하며, 가격 오차 또는 변동성 bracket 폭이 각 tolerance 이하가 되면 수렴한다. 무차익 범위 밖 가격, bracket 안에서 만들 수 없는 가격, 반복 횟수 안에 수렴하지 못한 경우는 임의의 IV 대신 명시적으로 실패한다.

In [1]:
from __future__ import annotations

import math
from numbers import Real
from pathlib import Path
import sys

import numpy as np
import pandas as pd

project_root_candidates = (Path.cwd(), *Path.cwd().parents)
PROJECT_ROOT = next(
    (path for path in project_root_candidates if (path / "pyproject.toml").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Could not locate the project root from the current directory")

src_path = str(PROJECT_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from option_pricing_volatility.models.bsm import bsm_price
from option_pricing_volatility.volatility import (
    estimate_forward_and_dividend_yield,
    implied_volatility,
)

## Synthetic round-trip

이 노트는 모듈에서 구현한 iv_calculation module이 올바르게 동작하는지
간단한 예시를 통해서 재구현한다. 

1. 알려진 변동성으로 call과 put의 BSM 가격을 만든다. 
2. 그 가격만을 target으로 사용해 IV를 복원한다.
    - IV복원 - 원래값 => volatility_error
3. 복원한 IV로 다시 BSM을 계산한다. 
    - 원래 BSM 가격 - 복원 IV 사용 BSM 가격 => repricing_error 



In [2]:
synthetic_parameters = {
    "spot": 100.0,
    "strike": 105.0,
    "maturity": 0.75,
    "rate": 0.03,
    "dividend_yield": 0.01,
}
original_volatility = 0.24
price_tolerance = 1e-8

round_trip_rows = []
for option_type in ("call", "put"):
    market_price = bsm_price(
        **synthetic_parameters,
        volatility=original_volatility,
        option_type=option_type,
    )
    result = implied_volatility(
        **synthetic_parameters,
        market_price=market_price,
        option_type=option_type,
        price_tolerance=price_tolerance,
    )
    repriced = bsm_price(
        **synthetic_parameters,
        volatility=result.volatility,
        option_type=option_type,
    )
    round_trip_rows.append(
        {
            "option_type": option_type,
            "original_volatility": original_volatility,
            "recovered_implied_volatility": result.volatility,
            "volatility_error": result.volatility - original_volatility,
            "repricing_error": repriced - market_price,
        }
    )

round_trip_df = pd.DataFrame(round_trip_rows)
assert round_trip_df["volatility_error"].abs().max() <= 1e-8
assert round_trip_df["repricing_error"].abs().max() <= price_tolerance
round_trip_df

,option_type,original_volatility,recovered_implied_volatility,volatility_error,repricing_error
0,call,0.24,0.24,-3.536954e-11,-1.210786e-09
1,put,0.24,0.24,-3.536954e-11,-1.210779e-09


## Fixed SPX processed snapshot

고정된 processed snapshot의 유효한 `bid`와 `ask`에서 이미 계산된 `mid`를 시장가격으로 사용한다. `last`와 provider의 IV는 대체값으로 사용하지 않는다.

무위험금리는 snapshot 기준일인 **2026-07-15**의 [FRED DGS1MO (1-Month Treasury Constant Maturity Rate)](https://fred.stlouisfed.org/series/DGS1MO) **3.73%**를 사용한다. DGS1MO CMT는 bond-equivalent yield이므로 [U.S. Treasury의 CMT convention 설명](https://home.treasury.gov/policy-issues/financing-the-government/interest-rate-statistics/interest-rates-frequently-asked-questions)에 따라 프로젝트 BSM이 요구하는 연속복리 연율로 변환한다.

$$
r = 2\log\left(1 + \frac{0.0373}{2}\right) = 0.0369564425
$$

한 snapshot과 한 expiry에 이 값 하나를 사용하며 별도의 금리곡선 보간은 하지 않는다. 동일 strike의 finite call/put `mid` pair마다 $F_i=K_i+e^{rT}(C_i-P_i)$를 계산하고 그 median을 대표 `forward`로 사용한다. 이어서 $q=r-\log(F/S)/T$로 연속복리 `dividend_yield` 하나를 추정한다. 이 계산은 package 함수에서 수행하고, 원본 processed CSV는 그대로 둔 채 별도의 enriched CSV를 생성한다.

In [3]:
SPX_PROCESSED_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "marketdata_spx"
    / "SPX_2026-07-15_dte030_pm_sl450_processed.csv"
)
SPX_MODEL_INPUTS_PATH = SPX_PROCESSED_PATH.with_name(
    "SPX_2026-07-15_dte030_pm_sl450_model_inputs.csv"
)
if not SPX_PROCESSED_PATH.is_file():
    raise FileNotFoundError(
        f"Expected the existing processed SPX snapshot at {SPX_PROCESSED_PATH}"
    )

spx_df = pd.read_csv(SPX_PROCESSED_PATH)
required_columns = {
    "underlying",
    "option_type",
    "strike",
    "spot",
    "T",
    "bid",
    "ask",
    "mid",
}

"""
불러온 SPX processed data가 실제로 이번 작업에 유효한지를 재검사한다.

1. 정의한 required_colums중에서 실제 pandas df에 없는 colum이 존재하는가?
    - 만약 있다면 => SPX schema에는 없는 열을 요구한다는 에러.
2. underlying == "SPX" 인지 검사한다. ( 즉, 기초자산이 모두 SPX임을 재확인 )
3. 매도호가 > 매수호가 > 0 인지 확인.
4. 허용오차 ( 절대오차 1e-12, 상대오차 0.0 을 기준) 에서,  2개의 값이 모두 유사한지 확인한다.
    - mid , bid+ask/2
"""
missing_columns = sorted(required_columns.difference(spx_df.columns))
if missing_columns:
    raise ValueError(f"Processed SPX schema is missing columns: {missing_columns}")
if not spx_df["underlying"].eq("SPX").all():
    raise ValueError("Processed snapshot contains a non-SPX underlying")
if not ((spx_df["bid"] > 0.0) & (spx_df["ask"] >= spx_df["bid"])).all():
    raise ValueError("Processed snapshot contains an invalid two-sided quote")
if not np.allclose(
    spx_df["mid"],
    (spx_df["bid"] + spx_df["ask"]) / 2.0,
    rtol=0.0,
    atol=1e-12,
):
    raise ValueError("mid does not match the bid/ask midpoint convention")

"""
위 불러온 데이터의 유효성 검사가 끝나면, 불러온 df 의 복사본을 생성, 원본을 남긴다.

복사본에 snapshot 공통 r/F/q와 provenance를 추가해 별도의 enriched CSV로 저장한다.
그 후 타겟가격과 시장가격을 모두 mid_price로 정의한다. ( 즉, 매도호가, 매수호가의 평균)
"""
spx_df = spx_df.copy()
dgs1mo_bond_equivalent_yield = 0.0373
risk_free_rate = 2.0 * math.log1p(dgs1mo_bond_equivalent_yield / 2.0)
forward_dividend_estimate = estimate_forward_and_dividend_yield(
    spx_df,
    risk_free_rate,
)

spx_df["risk_free_rate"] = risk_free_rate
spx_df["forward"] = forward_dividend_estimate.forward
spx_df["dividend_yield"] = forward_dividend_estimate.dividend_yield
spx_df["risk_free_rate_source"] = (
    "FRED DGS1MO 2026-07-15 3.73%; "
    "bond-equivalent converted with r=2*log(1+y/2)"
)
spx_df["forward_method"] = (
    "median same-strike put-call parity using call/put mid"
)
spx_df["dividend_yield_method"] = "q = r - log(F / S) / T"
spx_df.to_csv(SPX_MODEL_INPUTS_PATH, index=False)

spx_df["target_price"] = spx_df["mid"]
spx_df["market_price"] = spx_df["target_price"]
{
    "rows": len(spx_df),
    "risk_free_rate": risk_free_rate,
    "forward": forward_dividend_estimate.forward,
    "dividend_yield": forward_dividend_estimate.dividend_yield,
    "enriched_csv": SPX_MODEL_INPUTS_PATH.relative_to(PROJECT_ROOT),
}

{'rows': 424,
 'risk_free_rate': 0.036956442491546095,
 'forward': 7594.91863819129,
 'dividend_yield': 0.000845553274877453,
 'enriched_csv': PosixPath('data/processed/marketdata_spx/SPX_2026-07-15_dte030_pm_sl450_model_inputs.csv')}

In [4]:
"""
배당수익률, 무위험이자율 열이 df에 존재하는지 본다. 
"""
model_input_columns = ("risk_free_rate", "dividend_yield")
missing_model_columns = [
    column for column in model_input_columns if column not in spx_df.columns
]

"""
이 함수는 옵션체인 df 의 한행 ( 한행 = 하나의 옵션) 을 받아서, IV 계산결과 한행을 반환한다.

"""
def invert_spx_row(row: pd.Series) -> pd.Series:


    """
    입력검사1 : 무위험이자율, 수익률
    """
    if missing_model_columns:
        return pd.Series(
            {
                "implied_volatility": np.nan,
                "repricing_error": np.nan,
                "iv_iterations": np.nan,
                "iv_converged": False,
                "iv_status": "failed",
                "iv_failure_reason": (
                    "MISSING_MODEL_INPUT:" + ",".join(missing_model_columns)
                ),
            }
        )

    nonfinite_inputs = [
        column
        for column in model_input_columns
        if (
            isinstance(row[column], (bool, np.bool_))
            or not isinstance(row[column], Real)
            or not math.isfinite(row[column])
        )
    ]
    if nonfinite_inputs:
        return pd.Series(
            {
                "implied_volatility": np.nan,
                "repricing_error": np.nan,
                "iv_iterations": np.nan,
                "iv_converged": False,
                "iv_status": "failed",
                "iv_failure_reason": (
                    "NONFINITE_MODEL_INPUT:" + ",".join(nonfinite_inputs)
                ),
            }
        )

    try:
        result = implied_volatility(
            spot=row["spot"],
            strike=row["strike"],
            maturity=row["T"],
            rate=row["risk_free_rate"],
            market_price=row["market_price"],
            option_type=row["option_type"],
            dividend_yield=row["dividend_yield"],
        )
    except (ValueError, RuntimeError) as exc:
        return pd.Series(
            {
                "implied_volatility": np.nan,
                "repricing_error": np.nan,
                "iv_iterations": np.nan,
                "iv_converged": False,
                "iv_status": "failed",
                "iv_failure_reason": f"{type(exc).__name__}: {exc}",
            }
        )

    return pd.Series(
        {
            "implied_volatility": result.volatility,
            "repricing_error": result.repricing_error,
            "iv_iterations": result.iterations,
            "iv_converged": result.converged,
            "iv_status": "success",
            "iv_failure_reason": pd.NA,
        }
    )

iv_results = spx_df.apply(invert_spx_row, axis=1)
spx_iv_df = pd.concat([spx_df, iv_results], axis=1)
iv_run_summary = (
    spx_iv_df.groupby(["iv_status", "iv_failure_reason"], dropna=False)
    .size()
    .rename("row_count")
    .reset_index()
)
iv_run_summary

,iv_status,iv_failure_reason,row_count
0,failed,ValueError: market_price must lie within the d...,1
1,failed,ValueError: market_price must lie within the d...,1
2,failed,ValueError: market_price must lie within the d...,1
3,failed,ValueError: market_price must lie within the d...,1
4,failed,ValueError: market_price must lie within the d...,1
5,failed,ValueError: market_price must lie within the d...,1
6,failed,ValueError: market_price must lie within the d...,1
7,failed,ValueError: market_price must lie within the d...,1
8,failed,ValueError: market_price must lie within the d...,1
9,failed,ValueError: market_price must lie within the d...,1


## Plotting-ready tidy result

추정한 snapshot 공통 `forward`로 `log_forward_moneyness = log(strike / forward)`를 계산하고 그 값을 기준으로 정렬한다. IV inversion 실패 행도 원인과 함께 `iv_skew_df`에 그대로 남긴다.

In [5]:
if "forward" in spx_iv_df.columns:
    valid_forward = (
        np.isfinite(spx_iv_df["forward"]) & (spx_iv_df["forward"] > 0.0)
    )
    log_forward_moneyness = pd.Series(np.nan, index=spx_iv_df.index)
    log_forward_moneyness.loc[valid_forward] = np.log(
        spx_iv_df.loc[valid_forward, "strike"]
        / spx_iv_df.loc[valid_forward, "forward"]
    )
    spx_iv_df["log_forward_moneyness"] = log_forward_moneyness

tidy_columns = ["strike", "option_type", "market_price"]
tidy_columns.extend(
    column
    for column in (
        "forward",
        "spot_moneyness",
        "log_spot_moneyness",
        "log_forward_moneyness",
    )
    if column in spx_iv_df.columns
)
tidy_columns.extend(
    [
        "implied_volatility",
        "repricing_error",
        "iv_status",
        "iv_failure_reason",
    ]
)

x_axis_column = (
    "log_forward_moneyness"
    if "log_forward_moneyness" in spx_iv_df.columns
    else "strike"
)
iv_skew_df = (
    spx_iv_df.loc[:, tidy_columns]
    .sort_values([x_axis_column, "option_type"], kind="stable")
    .reset_index(drop=True)
)
iv_skew_df.head(10)

,strike,option_type,market_price,forward,spot_moneyness,log_spot_moneyness,log_forward_moneyness,implied_volatility,repricing_error,iv_status,iv_failure_reason
0,2400,call,5175.85,7594.918638,0.316940,-1.149043,-1.152011,NaN,NaN,failed,ValueError: market_price must lie within the d...
1,2600,call,4976.95,7594.918638,0.343352,-1.069000,-1.071968,NaN,NaN,failed,ValueError: market_price must lie within the d...
2,2800,call,4777.30,7594.918638,0.369763,-0.994892,-0.997860,NaN,NaN,failed,ValueError: market_price must lie within the d...
3,3000,call,4579.00,7594.918638,0.396175,-0.925899,-0.928867,NaN,NaN,failed,ValueError: market_price must lie within the d...
4,3000,put,0.10,7594.918638,0.396175,-0.925899,-0.928867,0.945559,6.419214e-09,success,NaN
5,3200,call,4378.60,7594.918638,0.422587,-0.861361,-0.864329,NaN,NaN,failed,ValueError: market_price must lie within the d...
6,3400,call,4180.25,7594.918638,0.448998,-0.800736,-0.803704,NaN,NaN,failed,ValueError: market_price must lie within the d...
7,3400,put,0.10,7594.918638,0.448998,-0.800736,-0.803704,0.822734,2.689410e-09,success,NaN
8,3600,call,3981.30,7594.918638,0.475410,-0.743578,-0.746546,NaN,NaN,failed,ValueError: market_price must lie within the d...
9,3600,put,0.15,7594.918638,0.475410,-0.743578,-0.746546,0.789386,-1.136236e-09,success,NaN
